# Feature/Shape GT Tracking Experiment

Fresh sandbox for dense adjacent-frame matching from GT segmentation masks and SpatialDINO features. The GT label IDs are used only for scoring after costs and Hungarian assignments are computed.

In [32]:
from pathlib import Path
import importlib.util
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch

module_path = repo_root / "scripts" / "evaluation" / "feature_shape_tracking_experiment.py"
if str(module_path.parent) not in sys.path:
    sys.path.insert(0, str(module_path.parent))
spec = importlib.util.spec_from_file_location("feature_shape_tracking_experiment", module_path)
experiment = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = experiment
spec.loader.exec_module(experiment)

ExperimentConfig = experiment.ExperimentConfig
aggregate_summary = experiment.aggregate_summary
run_adjacent_tracking_experiment = experiment.run_adjacent_tracking_experiment
save_result = experiment.save_result

print(f"repo_root: {repo_root}")
print(f"cuda: {torch.cuda.is_available()} devices={torch.cuda.device_count()}")

repo_root: /nfs/scratch2/inacio/code/llsm/spatialdino/spatialdino
cuda: True devices=1


In [43]:
# Edit these paths before running.
INPUT_PATH = Path("/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/rope/")
GT_SEGMENTATION_PATH = Path("/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/gt/")
OUTPUT_PATH = "/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/tests_rope/"

# For a fast first pass, set MAX_FRAMES=20 or MAX_ADJACENT_PAIRS=3.
MAX_FRAMES = None
MAX_ADJACENT_PAIRS = None

# Large centroid gate applied before Hungarian assignment. Set enabled=False for fully dense matching.
SEARCH_RADIUS_ENABLED = True
SEARCH_RADIUS_XY = 50.0
SEARCH_RADIUS_Z = 25.0

In [44]:
visible_cuda_devices = tuple(f"cuda:{idx}" for idx in range(torch.cuda.device_count()))
devices = visible_cuda_devices if visible_cuda_devices else ("cpu",)

config = ExperimentConfig(
    n_features=384,
    samples_per_object=128,
    n_feature_projections=64,
    n_shape_pairs=512,
    n_shape_quantiles=64,
    shape_weight=0.1,
    methods=("feature_mean", "sliced_wasserstein", "sliced_wasserstein_shape"),
    seed=12345,
    feature_channel_block=64,
    object_batch_size=256,
    sample_batch_size=131_072,
    cost_block_rows=1024,
    mask_workers=8,
    signature_workers=len(devices),
    devices=devices,
    pair_device=devices[0],
    use_float16_cost=False,
    torch_threads_per_worker=1,
    max_frames=MAX_FRAMES,
    max_adjacent_pairs=MAX_ADJACENT_PAIRS,
    search_radius_enabled=SEARCH_RADIUS_ENABLED,
    search_radius_xy=SEARCH_RADIUS_XY,
    search_radius_z=SEARCH_RADIUS_Z,
    progress=True,
)

config

ExperimentConfig(n_features=384, samples_per_object=128, n_feature_projections=64, n_shape_pairs=512, n_shape_quantiles=64, shape_weight=0.1, methods=('feature_mean', 'sliced_wasserstein', 'sliced_wasserstein_shape'), seed=12345, feature_channel_block=64, object_batch_size=256, sample_batch_size=131072, cost_block_rows=1024, mask_workers=8, signature_workers=1, devices=('cuda:0',), pair_device='cuda:0', use_float16_cost=False, torch_threads_per_worker=1, max_frames=None, max_adjacent_pairs=None, search_radius_enabled=True, search_radius_xy=50.0, search_radius_z=25.0, progress=True)

In [45]:
result = run_adjacent_tracking_experiment(
    INPUT_PATH,
    GT_SEGMENTATION_PATH,
    config=config,
)

aggregate = aggregate_summary(result.summary)
display(aggregate)

[feature-shape-tracking] found 13 frame(s); using 384/390 feature channel(s), 128 sample(s)/object, 64 projection(s)
[feature-shape-tracking] signature devices=['cuda:0']; pair_device=cuda:0
[feature-shape-tracking] centroid search radius enabled: xy=50, z=25
[feature-shape-tracking] preparing mask metadata for 13 frame(s) with 8 worker(s)


mask metadata:   0%|                                                                                          …

[feature-shape-tracking] mask metadata completed in 1.10s
[feature-shape-tracking] building feature signatures for 13 frame(s) on cuda:0


feature signatures:   0%|                                                                                     …

[feature-shape-tracking] feature signatures completed in 49.77s
[feature-shape-tracking] evaluating 12 adjacent frame pair(s) on cuda:0


...

44->48 mean:   0%|                                                                                            …

44->48 SW:   0%|                                                                                              …

44->48 shape:   0%|                                                                                           …

[feature-shape-tracking] adjacent pair evaluation completed in 0.79s; total 51.68s


,method,frame_pairs,trackable_count,hungarian_accuracy,top1_accuracy,mean_true_rank,median_pair_margin,mean_pair_margin,link_pred_count,link_tp,...,radius_pair_count,radius_allowed_count,radius_allowed_fraction,radius_relaxed_ref_count,radius_relaxed_ref_fraction,true_link_outside_radius_count,true_link_outside_radius_fraction,assigned_outside_radius_count,assigned_outside_radius_fraction,assigned_relaxed_ref_count
0,feature_mean,12,3600,0.729444,0.686111,1.836111,0.004958,-34444.447826,3600,2626,...,1080000,26418,0.024461,1,0.000278,157,0.043611,9,0.0025,1
1,sliced_wasserstein,12,3600,0.737778,0.683333,1.833056,0.000025,-34444.444407,3600,2656,...,1080000,26418,0.024461,1,0.000278,157,0.043611,9,0.0025,1
2,sliced_wasserstein_shape,12,3600,0.709167,0.643056,1.921667,0.066113,-156766.774713,3600,2553,...,1080000,26418,0.024461,1,0.000278,157,0.043611,9,0.0025,1


In [46]:
display(
    result.summary.sort_values(
        ["method", "ref_frame_index"],
        kind="mergesort",
    )
)

,method,ref_frame_index,cand_frame_index,ref_timepoint,cand_timepoint,ref_count,cand_count,trackable_count,hungarian_correct,hungarian_accuracy,...,assigned_outside_radius_count,assigned_outside_radius_fraction,assigned_relaxed_ref_count,link_pred_count,link_tp,link_fp,link_fn,link_precision,link_recall,link_f1
0,feature_mean,0,1,00,04,300,300,300,201,0.670000,...,0,0.000000,0,300,201,99,99,0.670000,0.670000,0.670000
3,feature_mean,1,2,04,08,300,300,300,211,0.703333,...,1,0.003333,0,300,211,89,89,0.703333,0.703333,0.703333
6,feature_mean,2,3,08,12,300,300,300,202,0.673333,...,1,0.003333,0,300,202,98,98,0.673333,0.673333,0.673333
9,feature_mean,3,4,12,16,300,300,300,213,0.710000,...,1,0.003333,0,300,213,87,87,0.710000,0.710000,0.710000
12,feature_mean,4,5,16,20,300,300,300,235,0.783333,...,0,0.000000,0,300,235,65,65,0.783333,0.783333,0.783333
15,feature_mean,5,6,20,24,300,300,300,224,0.746667,...,1,0.003333,0,300,224,76,76,0.746667,0.746667,0.746667
18,feature_mean,6,7,24,28,300,300,300,189,0.630000,...,0,0.000000,0,300,189,111,111,0.630000,0.630000,0.630000
21,feature_mean,7,8,28,32,300,300,300,214,0.713333,...,0,0.000000,0,300,214,86,86,0.713333,0.713333,0.713333
24,feature_mean,8,9,32,36,300,300,300,214,0.713333,...,1,0.003333,0,300,214,86,86,0.713333,0.713333,0.713333
27,feature_mean,9,10,36,40,300,300,300,254,0.846667,...,2,0.006667,0,300,254,46,46,0.846667,0.846667,0.846667


In [47]:
display(
    result.link_metrics.sort_values(
        ["method", "ref_frame_index"],
        kind="mergesort",
    )
)

,method,ref_frame_index,cand_frame_index,ref_timepoint,cand_timepoint,ref_count,cand_count,trackable_count,link_pred_count,link_tp,...,radius_pair_count,radius_allowed_count,radius_allowed_fraction,radius_relaxed_ref_count,radius_relaxed_ref_fraction,true_link_outside_radius_count,true_link_outside_radius_fraction,assigned_outside_radius_count,assigned_outside_radius_fraction,assigned_relaxed_ref_count
0,feature_mean,0,1,00,04,300,300,300,300,201,...,90000,2441,0.027122,0,0.000000,13,0.043333,0,0.000000,0
3,feature_mean,1,2,04,08,300,300,300,300,211,...,90000,2395,0.026611,0,0.000000,16,0.053333,1,0.003333,0
6,feature_mean,2,3,08,12,300,300,300,300,202,...,90000,2325,0.025833,0,0.000000,10,0.033333,1,0.003333,0
9,feature_mean,3,4,12,16,300,300,300,300,213,...,90000,2271,0.025233,0,0.000000,16,0.053333,1,0.003333,0
12,feature_mean,4,5,16,20,300,300,300,300,235,...,90000,2231,0.024789,0,0.000000,8,0.026667,0,0.000000,0
15,feature_mean,5,6,20,24,300,300,300,300,224,...,90000,2187,0.024300,0,0.000000,11,0.036667,1,0.003333,0
18,feature_mean,6,7,24,28,300,300,300,300,189,...,90000,2139,0.023767,0,0.000000,18,0.060000,0,0.000000,0
21,feature_mean,7,8,28,32,300,300,300,300,214,...,90000,2105,0.023389,0,0.000000,16,0.053333,0,0.000000,0
24,feature_mean,8,9,32,36,300,300,300,300,214,...,90000,2072,0.023022,0,0.000000,17,0.056667,1,0.003333,0
27,feature_mean,9,10,36,40,300,300,300,300,254,...,90000,2073,0.023033,0,0.000000,9,0.030000,2,0.006667,0


In [48]:
bad_links = result.assignments[result.assignments["is_true_link"] == False]
display(
    bad_links.sort_values(
        ["method", "ref_frame_index", "cost"],
        kind="mergesort",
    ).head(100)
)

,method,ref_frame_index,cand_frame_index,ref_timepoint,cand_timepoint,ref_label,assigned_cand_label,cost,is_true_link,inside_search_radius,radius_relaxed_for_ref,centroid_delta_y,centroid_delta_x,centroid_delta_z,centroid_distance_xy
187,feature_mean,0,1,00,04,188,161,0.010708,False,True,False,-41.871857,-0.572876,1.532661,41.875775
84,feature_mean,0,1,00,04,85,23,0.010820,False,True,False,-34.666656,22.716675,2.900002,41.446645
162,feature_mean,0,1,00,04,163,268,0.010824,False,True,False,17.465736,16.732880,-2.690910,24.187625
242,feature_mean,0,1,00,04,243,193,0.011748,False,True,False,39.481796,-2.952377,5.240898,39.592029
153,feature_mean,0,1,00,04,154,120,0.012258,False,True,False,40.765625,-8.859375,0.125000,41.717199
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20,feature_mean,0,1,00,04,21,121,0.077238,False,True,False,40.442322,10.442322,-5.557693,41.768690
131,feature_mean,0,1,00,04,132,15,0.080964,False,True,False,49.625000,11.875000,-13.791667,51.026035
140,feature_mean,0,1,00,04,141,71,0.105595,False,True,False,-6.770832,28.187500,-15.729167,28.989297
104,feature_mean,0,1,00,04,105,85,0.130002,False,True,False,-46.661530,4.769226,21.171429,46.904625


In [49]:
for method, tracks_df in result.tracks.items():
    print(method)
    display(tracks_df.head(40))

feature_mean


,track_id,start,t,x,y,z,A,track_length
0,1,1,1,309.000000,223.000000,12.000000,176.152542,13
1,1,1,2,316.666656,224.079361,12.460318,179.301590,13
2,1,1,3,360.847443,206.966095,10.932203,165.050842,13
3,1,1,4,363.460327,200.174606,8.079365,164.555557,13
4,1,1,5,360.687500,206.578125,9.703125,162.390625,13
5,1,1,6,366.033325,201.966660,9.283334,165.716660,13
6,1,1,7,358.609375,199.078125,14.562500,164.062500,13
7,1,1,8,366.813568,197.271179,16.779661,163.050842,13
8,1,1,9,367.034485,193.810349,17.068966,164.793106,13
9,1,1,10,380.477600,188.014923,12.477612,162.865677,13


sliced_wasserstein


,track_id,start,t,x,y,z,A,track_length
0,1,1,1,309.000000,223.000000,12.000000,176.152542,13
1,1,1,2,316.666656,224.079361,12.460318,179.301590,13
2,1,1,3,360.847443,206.966095,10.932203,165.050842,13
3,1,1,4,363.460327,200.174606,8.079365,164.555557,13
4,1,1,5,360.687500,206.578125,9.703125,162.390625,13
5,1,1,6,366.033325,201.966660,9.283334,165.716660,13
6,1,1,7,358.609375,199.078125,14.562500,164.062500,13
7,1,1,8,366.813568,197.271179,16.779661,163.050842,13
8,1,1,9,367.034485,193.810349,17.068966,164.793106,13
9,1,1,10,380.477600,188.014923,12.477612,162.865677,13


sliced_wasserstein_shape


,track_id,start,t,x,y,z,A,track_length
0,1,1,1,309.000000,223.000000,12.000000,176.152542,13
1,1,1,2,316.666656,224.079361,12.460318,179.301590,13
2,1,1,3,309.758057,214.370972,9.225806,174.725800,13
3,1,1,4,286.432831,225.343277,16.179104,173.582092,13
4,1,1,5,252.273285,187.763977,38.329193,153.338516,13
5,1,1,6,218.003113,189.408096,49.137074,155.112152,13
6,1,1,7,214.258957,220.432510,58.691460,154.977966,13
7,1,1,8,226.601974,236.447372,50.611843,151.875000,13
8,1,1,9,233.902512,235.823898,51.006290,152.172958,13
9,1,1,10,227.348831,235.561050,51.543606,151.563950,13


In [50]:
failures = result.per_object[result.per_object["is_correct"] == False]
display(
    failures.sort_values(
        ["method", "margin", "true_rank"],
        ascending=[True, True, False],
        kind="mergesort",
    ).head(100)
)

,method,ref_frame_index,cand_frame_index,ref_timepoint,cand_timepoint,ref_label,true_cand_label,assigned_cand_label,is_correct,true_rank,true_cost,best_false_cost,margin,true_inside_search_radius,assigned_inside_search_radius,radius_relaxed_for_ref,true_centroid_delta_y,true_centroid_delta_x,true_centroid_delta_z,true_centroid_distance_xy
9231,feature_mean,10,11,40,44,232,232,284,False,11,1.000000e+06,0.013849,-1.000000e+06,False,True,False,-12.250512,54.983948,-0.168137,56.332136
9958,feature_mean,11,12,44,48,59,59,146,False,5,1.000000e+06,0.014588,-1.000000e+06,False,True,False,-10.354767,-57.777863,0.033360,58.698404
8255,feature_mean,9,10,36,40,156,156,238,False,8,1.000000e+06,0.017627,-1.000000e+06,False,True,False,-11.820248,78.183075,14.052374,79.071559
8268,feature_mean,9,10,36,40,169,169,300,False,7,1.000000e+06,0.017909,-1.000000e+06,False,True,False,64.385895,-2.595184,0.083242,64.438175
9107,feature_mean,10,11,40,44,108,108,168,False,9,1.000000e+06,0.019561,-1.000000e+06,False,True,False,71.058189,21.228394,0.055146,74.161385
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7250,feature_mean,8,9,32,36,51,51,247,False,5,1.000000e+06,0.032443,-1.000000e+06,False,True,False,68.151520,-72.545441,3.303030,99.536278
1993,feature_mean,2,3,08,12,194,194,182,False,5,1.000000e+06,0.032658,-1.000000e+06,False,True,False,-3.960724,-58.818462,-1.983858,58.951665
2855,feature_mean,3,4,12,16,156,156,159,False,9,1.000000e+06,0.032739,-1.000000e+06,False,True,False,-62.148346,-0.410023,-26.477901,62.149698
7312,feature_mean,8,9,32,36,113,113,123,False,5,1.000000e+06,0.033447,-1.000000e+06,False,True,False,15.611366,-64.019402,-4.522072,65.895360


In [51]:
display(result.timings)

,stage,seconds,ref_frame_index,cand_frame_index,ref_timepoint,cand_timepoint,method,cost_seconds,assignment_and_scoring_seconds
0,metadata,1.096276,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,signatures,49.773370,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,pair_costs_and_assignment,0.793387,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,total,51.679513,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,0.0,1.0,00,04,feature_mean,0.036566,0.006421
5,NaN,NaN,0.0,1.0,00,04,sliced_wasserstein,0.036566,0.005653
6,NaN,NaN,0.0,1.0,00,04,sliced_wasserstein_shape,0.036566,0.005551
7,NaN,NaN,1.0,2.0,04,08,feature_mean,0.033439,0.006251
8,NaN,NaN,1.0,2.0,04,08,sliced_wasserstein,0.033439,0.006186
9,NaN,NaN,1.0,2.0,04,08,sliced_wasserstein_shape,0.033439,0.005776


In [52]:
saved_paths = save_result(result, OUTPUT_PATH)
saved_paths

{'summary_csv': '/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/tests_rope/summary.csv',
 'aggregate_summary_csv': '/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/tests_rope/aggregate_summary.csv',
 'per_object_csv': '/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/tests_rope/per_object.csv',
 'assignments_csv': '/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/tests_rope/assignments.csv',
 'link_metrics_csv': '/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/tests_rope/link_metrics.csv',
 'track_points_csv': '/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_gap/n_300/gap_3/tests_rope/track_points_debug.csv',
 'timings_csv': '/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/fast_g